# CLIP on Fruits 360
**Model:** OpenAI CLIP ViT-B/32 &nbsp;|&nbsp; **Dataset:** Fruits 360

Demonstrates:
1. **Zero-shot classification** — no labels, just natural language descriptions
2. **Prompt engineering** — how template wording affects accuracy
3. **Prompt ensembling** — averaging templates for robustness
4. **Image–text similarity matrix** — visualising CLIP's joint embedding space
5. **Text-to-image retrieval** — query with descriptive phrases
6. **Image-to-image retrieval** — visual similarity without text
7. **CLIP failure mode** — fine-grained citrus confusion motivates DINOv2 + metric learning

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import clip
import torch
import torch.nn.functional as F
import pathlib
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.manifold import TSNE

In [ ]:
%cd ~/cvi4ic-notebooks/11
from clip_utils import discover_classes, suggest_folders, build_fruit_map, load_class_images

In [ ]:
# ── Dataset & device config ──────────────────────────────────────────────────
# Fruits 360 (original size) is stored at the path below.
# Expected layout: DATASET_ROOT/<ClassName>/<image>.jpg
DATASET_ROOT = pathlib.Path("../fruits-360-100x100/Training")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert DATASET_ROOT.exists(), (
    f"Dataset root not found: {DATASET_ROOT}\n"
    "Please update DATASET_ROOT to the directory that contains the class folders."
)
all_classes = discover_classes(DATASET_ROOT)
print(f"Device:              {DEVICE}")
print(f"Dataset root:        {DATASET_ROOT}")
print(f"Total classes found: {len(all_classes)}")

In [ ]:
# ── Diagnostic: inspect actual folder names ──────────────────────────────────
# If build_fruit_map warns about missing prefixes, run this cell to see
# what the dataset actually contains and adjust FRUIT_KEYWORDS accordingly.
print("First 10 class folders found:")
for c in all_classes[:10]:
    print(f"  {c}")

# Uncomment to search for a specific fruit:
# suggest_folders("apple", all_classes)

## 1. Load CLIP

In [ ]:
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()

res     = model.visual.input_resolution
n_patch = (res // 32) ** 2
print(f"Model:              CLIP ViT-B/32")
print(f"Input resolution:   {res}×{res} px")
print(f"Patch size:         32×32 px  →  {n_patch} patches per image")
print(f"Embedding dim:      {model.visual.output_dim}  (shared image+text space)")
print(f"Text context:       {model.context_length} tokens (BPE)")
print(f"Logit scale (1/τ):  {model.logit_scale.exp().item():.2f}")
total = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Total parameters:   {total:.1f}M")

## 2. Load Fruits 360 Images

We pick 10 visually distinct fruit categories. Folder names are mapped to clean labels used in CLIP text prompts.

In [ ]:
N_PER_CLASS = 20

# (folder_prefix, clean_label_for_prompts)
FRUIT_KEYWORDS = [
    ("Apple Golden",  "apple"),
    ("Banana 1",      "banana"),
    ("Cherry 1",      "cherry"),
    ("Grape Blue",    "grape"),
    ("Kiwi",          "kiwi"),
    ("Lemon 1",       "lemon"),
    ("Mango 1",       "mango"),
    ("Orange 1",      "orange"),
    ("Pineapple",     "pineapple"),
    ("Strawberry 1",  "strawberry"),
]

FRUIT_MAP = build_fruit_map(FRUIT_KEYWORDS, all_classes, DATASET_ROOT)

class_images: dict[str, list] = {
    folder: load_class_images(folder, DATASET_ROOT, n=N_PER_CLASS)
    for folder in FRUIT_MAP
}

class_folders = list(FRUIT_MAP.keys())
class_labels  = list(FRUIT_MAP.values())

print("Loaded classes:")
for folder, label in FRUIT_MAP.items():
    print(f"  {folder:<25} → '{label}'  ({len(class_images[folder])} images)")

In [ ]:
# Preview: one image per class
cols = 5
rows = (len(FRUIT_MAP) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.8))
for ax in axes.flat:
    ax.axis("off")
for ax, (folder, label) in zip(axes.flat, FRUIT_MAP.items()):
    ax.imshow(class_images[folder][0])
    ax.set_title(f"{label}\n({folder})", fontsize=8)
    ax.axis("off")
plt.suptitle("One image per class — Fruits 360", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Zero-Shot Classification

CLIP classifies images **without any training data** by comparing image embeddings to text prompt embeddings.

```
query image  →  image encoder  →  z_img  ┐
                                           ├─ argmax cosine_sim  →  predicted class
"a photo of a {class}." → text encoder → z_txt  ┘
```

The text encoder acts as a **dynamic classifier head** — redefine the class list in natural language and the classifier changes instantly.

In [ ]:
from clip_utils import encode_text_prompts, classify_images

In [ ]:
# Encode class descriptions with a single template
TEMPLATE_SINGLE = "a photo of a {}."

prompts    = [TEMPLATE_SINGLE.format(lbl) for lbl in class_labels]
text_feats = encode_text_prompts(prompts, model, DEVICE)  # (C, 512)

print("Text prompts:")
for lbl, p in zip(class_labels, prompts):
    print(f"  {lbl:<12} → '{p}'")

In [ ]:
# Per-class accuracy
class_acc = {}
for true_idx, (folder, label) in enumerate(FRUIT_MAP.items()):
    # TODO

overall = sum(class_acc.values()) / len(class_acc)
print(f"Zero-shot accuracy — template: '{TEMPLATE_SINGLE}'")
print(f"  Overall: {overall:.1%}\n")

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2ecc71" if v >= 0.8 else "#e67e22" if v >= 0.5 else "#e74c3c"
          for v in class_acc.values()]
ax.barh(list(class_acc.keys()), list(class_acc.values()), color=colors)
ax.axvline(overall, color="navy", linestyle="--", label=f"Mean {overall:.1%}")
ax.set_xlim(0, 1.1)
ax.set_xlabel("Accuracy")
ax.set_title(f"CLIP Zero-Shot Accuracy per Class\n(template: '{TEMPLATE_SINGLE}')")
ax.legend()
for i, (lbl, v) in enumerate(class_acc.items()):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Prompt Engineering

The wording of the text prompt matters. Fruits 360 images have a **white background** — prompts that mention this may better match the training distribution.

We compare seven templates and measure accuracy on all 10 classes.

In [ ]:
from clip_utils import encode_template, gallery_accuracy

In [ ]:
TEMPLATES = {
    "bare":             "{}",
    # TODO
}

template_feats = {
    name: encode_template(tmpl, class_labels, model, DEVICE)
    for name, tmpl in TEMPLATES.items()
}
template_accs = {
    name: gallery_accuracy(class_images, class_folders, feats, model, preprocess, DEVICE)
    for name, feats in template_feats.items()
}

print(f"{'Template':<25}  Accuracy")
print("-" * 45)
for name, acc in sorted(template_accs.items(), key=lambda x: -x[1]):
    bar = "█" * int(acc * 30)
    print(f"  {name:<23}  {acc:.1%}  {bar}")

In [ ]:
# Visualise per-image predictions for each template on one class
VIZ_CLASS  = "apple"
VIZ_FOLDER = next(f for f, l in FRUIT_MAP.items() if l == VIZ_CLASS)
VIZ_IMAGES = class_images[VIZ_FOLDER][:5]

n_tmpl = len(TEMPLATES)
fig, axes = plt.subplots(n_tmpl, 5, figsize=(11, 1.7 * n_tmpl), squeeze=False)
for t_idx, (tname, _) in enumerate(TEMPLATES.items()):
    t_feats = template_feats[tname]
    for col, img in enumerate(VIZ_IMAGES):
        ax   = axes[t_idx, col]
        prbs = classify_images([img], t_feats, model, preprocess, DEVICE)[0]
        pred = class_labels[prbs.argmax()]
        ok   = pred == VIZ_CLASS
        ax.imshow(img)
        ax.set_title(f"{pred}\n{prbs.max():.2f}", fontsize=6.5,
                     color="green" if ok else "red", pad=2)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(tname, fontsize=7.5, rotation=0, labelpad=65, va="center")

plt.suptitle(f"Template effect on '{VIZ_CLASS}' images  (green = correct)", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Prompt Ensembling

Average text embeddings over multiple templates, then re-normalise. This is a **free accuracy boost** — from the CLIP paper: ensembling gave +3.5% on ImageNet zero-shot over the single best template.

```python
# For each class:
embeddings = [encode(template.format(class)) for template in templates]
ensemble   = F.normalize(mean(embeddings), dim=-1)
```

In [ ]:
from clip_utils import build_ensemble

In [ ]:
ensemble_feats = build_ensemble(TEMPLATES, class_labels, model, DEVICE)
ensemble_acc   = gallery_accuracy(class_images, class_folders, ensemble_feats, model, preprocess, DEVICE)

best_name = max(template_accs, key=template_accs.get)
print(f"Best single template: '{best_name}'  →  {template_accs[best_name]:.1%}")
print(f"Ensemble ({len(TEMPLATES)} templates):     →  {ensemble_acc:.1%}")
delta = ensemble_acc - template_accs[best_name]
print(f"Improvement:                           {'+'if delta >= 0 else ''}{delta:.1%}")

# Bar chart
all_results = dict(sorted(template_accs.items(), key=lambda x: x[1]))
all_results["── ensemble ──"] = ensemble_acc

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ["#3498db"] * len(template_accs) + ["#e74c3c"]
ax.barh(list(all_results.keys()), list(all_results.values()), color=bar_colors)
ax.set_xlim(0, 1.1)
ax.set_xlabel("Zero-shot accuracy")
ax.set_title("Prompt Template vs. Ensemble — Zero-Shot Accuracy")
for i, v in enumerate(all_results.values()):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Image–Text Similarity Matrix

The N×N cosine similarity matrix shows how well CLIP aligns each image with each class description. The **diagonal should be brightest** — each image should be most similar to its own class prompt.

This is the same structure as CLIP's training objective: maximise diagonal, minimise off-diagonal.

In [ ]:
from clip_utils import encode_images_batched

In [ ]:
# One representative image per class
sample_imgs = [class_images[folder][0] for folder in class_folders]
n_cls = len(class_labels)

img_feats  = # TODO
sim_matrix = # TODO

# Thumbnail strip + heatmap side by side
THUMB = 64
strip = np.zeros((n_cls * THUMB, THUMB, 3), dtype=np.uint8)
for i, img in enumerate(sample_imgs):
    strip[i * THUMB : (i + 1) * THUMB] = np.array(img.resize((THUMB, THUMB)))

fig = plt.figure(figsize=(13, 9))
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 5], wspace=0.04)
ax_thumb = fig.add_subplot(gs[0])
ax_heat  = fig.add_subplot(gs[1])

ax_thumb.imshow(strip)
ax_thumb.set_yticks([THUMB // 2 + i * THUMB for i in range(n_cls)])
ax_thumb.set_yticklabels(class_labels, fontsize=9)
ax_thumb.set_xticks([])
ax_thumb.set_title("Image", fontsize=9)

sns.heatmap(sim_matrix, ax=ax_heat,
            xticklabels=class_labels,
            yticklabels=[],
            annot=True, fmt=".2f",
            cmap="RdYlGn", vmin=-0.3, vmax=0.5,
            linewidths=0.4,
            cbar_kws={"label": "cosine similarity"})
ax_heat.set_title(
    "CLIP Image–Text Cosine Similarity\n"
    f"(rows = images,  cols = text prompts using template: '{TEMPLATE_SINGLE}')",
    fontsize=10)
ax_heat.set_xlabel("Text prompt class")
plt.setp(ax_heat.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Text-to-Image Retrieval

Encode a **natural language query** and retrieve the gallery images with the highest cosine similarity. No labels, no training — the same shared embedding space used for classification.

We build a gallery of 200 images (20 per class), then query with descriptive phrases that don't literally name the fruit.

In [ ]:
from clip_utils import build_gallery, text_retrieve

In [ ]:
gallery_imgs, gallery_labels, gallery_feats = build_gallery(
    class_images, FRUIT_MAP, model, preprocess, DEVICE
)
print(f"Gallery: {len(gallery_imgs)} images  ({len(FRUIT_MAP)} classes × {N_PER_CLASS} images)")

In [ ]:
TEXT_QUERIES = [
    # TODO
]

TOP_K = 5
fig, axes = plt.subplots(len(TEXT_QUERIES), TOP_K + 1,
                         figsize=(14, 2.0 * len(TEXT_QUERIES)))

for row, query in enumerate(TEXT_QUERIES):
    top_idx, sims = text_retrieve(query, gallery_feats, model, DEVICE, top_k=TOP_K)

    ax = axes[row, 0]
    ax.text(0.5, 0.5, f'"{query}"',
            ha="center", va="center", fontsize=8.5, style="italic",
            wrap=True, transform=ax.transAxes)
    ax.set_facecolor("#eef4fb")
    ax.axis("off")

    for col, (idx, sim) in enumerate(zip(top_idx, sims), start=1):
        ax = axes[row, col]
        ax.imshow(gallery_imgs[idx])
        ax.set_title(f"{gallery_labels[idx]}\n{sim:.3f}", fontsize=7)
        ax.axis("off")

plt.suptitle("Text-to-Image Retrieval with CLIP", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Image-to-Image Retrieval

CLIP image embeddings can be used for **pure visual similarity search** — no text at all. The same gallery index built above works directly.

In [ ]:
from clip_utils import image_retrieve

In [ ]:
QUERY_LABELS = ["apple", "banana", "grape", "strawberry"]
TOP_K_IMG    = 6

fig, axes = plt.subplots(len(QUERY_LABELS), TOP_K_IMG + 1,
                         figsize=(14, 2.1 * len(QUERY_LABELS)))

for row, qlabel in enumerate(QUERY_LABELS):
    qfolder   = next(f for f, l in FRUIT_MAP.items() if l == qlabel)
    query_img = class_images[qfolder][-1]   # last image of class
    top_idx, sims = image_retrieve(query_img, gallery_feats, model, preprocess, DEVICE, top_k=TOP_K_IMG)

    ax = axes[row, 0]
    ax.imshow(query_img)
    ax.set_title(f"QUERY\n{qlabel}", fontsize=8, fontweight="bold", color="navy")
    ax.axis("off")

    for col, (idx, sim) in enumerate(zip(top_idx, sims), start=1):
        ax = axes[row, col]
        ax.imshow(gallery_imgs[idx])
        # Top-1 is often the query image itself — expected behaviour
        ax.set_title(f"{gallery_labels[idx]}\n{sim:.3f}", fontsize=7,
                     color="#2ecc71" if gallery_labels[idx] == qlabel else "#e74c3c")
        ax.axis("off")

plt.suptitle("Image-to-Image Retrieval with CLIP  (green = same class, red = different)",
             fontsize=11)
plt.tight_layout()
plt.show()

## 9. CLIP Failure Mode — Fine-Grained Citrus

CLIP excels at **broad category** distinctions (apple vs banana vs pineapple). It struggles with **fine-grained visual differences** between similar varieties — precisely because its training signal is language, and text descriptions of similar fruits are nearly identical.

We test CLIP on five citrus varieties that are difficult to distinguish by text description alone. This is the motivation for **DINOv2 + metric learning** (see `fruits360_dinov3_triplet.ipynb`).

In [ ]:
CITRUS_KEYWORDS = [
    ("Orange 1",      "orange (var. 1)"),
    ("Orange 2",      "orange (var. 2)"),
    ("Orange 3",      "orange (var. 3)"),
    ("Mandarine 1",   "mandarine"),
    ("Clementine 1",  "clementine"),
]
N_CITRUS = 30

CITRUS_MAP = build_fruit_map(CITRUS_KEYWORDS, all_classes, DATASET_ROOT)

citrus_images = {
    folder: load_class_images(folder, DATASET_ROOT, n=N_CITRUS)
    for folder in CITRUS_MAP
}
citrus_folders = list(CITRUS_MAP.keys())
citrus_labels  = list(CITRUS_MAP.values())

# Preview
fig, axes = plt.subplots(1, len(CITRUS_MAP), figsize=(13, 3))
for ax, (folder, label) in zip(axes, CITRUS_MAP.items()):
    ax.imshow(citrus_images[folder][0])
    ax.set_title(f"{label}\n({folder})", fontsize=8)
    ax.axis("off")
plt.suptitle("Fine-grained citrus — can CLIP distinguish these?", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Zero-shot with variety-specific prompts
CITRUS_PROMPTS = [
    "a photo of an orange.",
    "a photo of an orange.",
    "a photo of an orange.",
    "a photo of a mandarine.",
    "a photo of a clementine.",
]

citrus_tfeats = encode_text_prompts(CITRUS_PROMPTS, model, DEVICE)

# Build confusion matrix
n_c  = len(CITRUS_MAP)
conf = np.zeros((n_c, n_c), dtype=int)
for true_idx, folder in enumerate(citrus_folders):
    probs = classify_images(citrus_images[folder], citrus_tfeats, model, preprocess, DEVICE)
    for pred in probs.argmax(axis=1):
        conf[true_idx, pred] += 1

conf_pct = conf / conf.sum(axis=1, keepdims=True) * 100
mean_acc = np.diag(conf_pct).mean()

short_labels = [l.replace("orange ", "orange\n") for l in citrus_labels]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(conf, ax=axes[0],
            xticklabels=short_labels, yticklabels=short_labels,
            annot=True, fmt="d", cmap="Blues",
            linewidths=0.5)
axes[0].set_title("Confusion Matrix — counts")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
plt.setp(axes[0].get_xticklabels(), rotation=35, ha="right", fontsize=8)
plt.setp(axes[0].get_yticklabels(), fontsize=8)

sns.heatmap(conf_pct, ax=axes[1],
            xticklabels=short_labels, yticklabels=short_labels,
            annot=True, fmt=".0f", cmap="RdYlGn",
            vmin=0, vmax=100, linewidths=0.5)
axes[1].set_title(f"Confusion Matrix — %\n(mean accuracy: {mean_acc:.1f}%)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
plt.setp(axes[1].get_xticklabels(), rotation=35, ha="right", fontsize=8)
plt.setp(axes[1].get_yticklabels(), fontsize=8)

plt.suptitle("CLIP on Fine-Grained Citrus Classes", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Mean accuracy on fine-grained citrus: {mean_acc:.1f}%")
print("(Compare to ~90%+ on the 10 distinct classes above)")

## Summary

| Task | CLIP performance | Why |
|---|---|---|
| Zero-shot (10 distinct classes) | ✅ High | Broad visual differences expressible in text |
| Prompt engineering | ✅ Context matters | "white background" template matches Fruits 360 distribution |
| Prompt ensembling | ✅ Free +% | Average embeddings smooth over template variance |
| Text-to-image retrieval | ✅ Works well | Joint embedding space; no training needed |
| Image-to-image retrieval | ✅ Works well | Image encoder alone; no text needed |
| Fine-grained citrus | ❌ Confusion | Visual differences between varieties not encoded in language |

### When to use CLIP vs. DINO

```
Query is text?                       → CLIP
Open-vocabulary / unseen classes?    → CLIP
Fine-grained visual discrimination?  → DINO + metric learning head
Dense prediction (segmentation)?     → DINO patch tokens
```